# КТ-1: корпус изображений городских камер и золотой набор

## Постановка задачи

Имеется **500 изображений городских камер наблюдения**, каждое из которых относится к уникальной камере. 
Предварительно, каждое изображение было вручную размечено людьми на предмет наличия определенных объектов и условий.
Затем, для каждого изображения был сгенерировано описание с использованием producer + verifier LLM пайплайна. При этом, сохраняется градация уверенности ответов модели и используются только проверенные факты.

Что требуется для последующего RAG-поиска:
пользователь задаёт вопрос об объекте или ситуации, а система возвращает
релевантные изображения по совпадению вопроса с описанием.

Ontology содержит **136 labels**:

- **base labels** — положение камеры, тип сцены, время суток, погода и качество;
- **detailed labels** — дорожная ситуация, инфраструктура, здания и дополнительные
  объекты.

В pilot 500 наблюдались 43 base и 92 detailed labels; одна категория ontology не
встретилась.

Цели контрольной точки:

1. зафиксировать исходный, ещё не очищенный и не чанкированный корпус;
2. выполнить EDA длин, шума, дубликатов и тематик;
3. создать gold-набор из 50 вопросов с validation/test split;
4. обосновать неприменимость No-RAG baseline для закрытого корпуса изображений;
5. измерить воспроизводимый retrieval baseline с доступом к корпусу;
6. документировать происхождение разметки и ограничения.

## 0. Установка и запуск локально или в Google Colab

Следующая ячейка устанавливает пакет проекта вместе со всеми зависимостями. Если notebook запущен из локального checkout репозитория, используется локальная версия в editable-режиме. В остальных средах, включая чистую сессию Google Colab, пакет устанавливается из ветки [`v0`](https://github.com/sisgrad/e2e_rag_gigaschool_llm_course/tree/v0) на GitHub.

После установки notebook не зависит от текущей рабочей директории или абсолютного пути к репозиторию. Пакет `gigaschool_rag` автоматически:

1. находит локальный `hf/v0` при запуске в контейнере или checkout репозитория;
2. в другой среде, включая Google Colab, загружает v0 из `mairon2300/video_retrieval` на Hugging Face.

Изображения используются для примеров только тогда, когда локальный каталог `hf/images/pilot-500-v1` доступен. Для анализа уже сохранённой разметки API-ключи не нужны.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/sisgrad/e2e_rag_gigaschool_llm_course"
REPOSITORY_REF = "v0"


def find_local_repository():
    """Найти checkout проекта независимо от текущей рабочей директории."""
    for directory in (Path.cwd(), *Path.cwd().parents):
        if (
            (directory / "pyproject.toml").is_file()
            and (directory / "src/gigaschool_rag").is_dir()
        ):
            return directory
    return None


local_repository = find_local_repository()
install_target = (
    str(local_repository)
    if local_repository is not None
    else f"git+{REPOSITORY_URL}.git@{REPOSITORY_REF}"
)

print(
    f"Установка локального checkout: {local_repository}"
    if local_repository is not None
    else f"Установка ветки {REPOSITORY_REF} из GitHub: {REPOSITORY_URL}"
)
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", "--editable", install_target]
    if local_repository is not None
    else [sys.executable, "-m", "pip", "install", "--quiet", install_target]
)
print("Пакет проекта и зависимости установлены.")

In [ ]:
import html
import inspect
from pathlib import Path

from IPython.display import HTML, display
from gigaschool_rag.versions.v0 import (
    resolve_dataset_root,
    resolve_image_root,
)


def show_package_source(*functions):
    """Показать исходный код package-функций в компактном раскрываемом блоке."""
    blocks = []
    for function in functions:
        qualified_name = f"{function.__module__}.{function.__name__}"
        source = html.escape(inspect.getsource(function))
        blocks.append(
            f"<details><summary><code>{qualified_name}</code></summary>"
            f"<pre><code>{source}</code></pre></details>"
        )
    display(HTML("\n".join(blocks)))


show_package_source(resolve_dataset_root, resolve_image_root)
DATASET_ROOT = resolve_dataset_root()
IMAGE_ROOT = resolve_image_root(DATASET_ROOT)

print(f"Dataset: {DATASET_ROOT}")
print(f"Images: {IMAGE_ROOT if IMAGE_ROOT else 'не установлены; visual preview будет пропущен'}")

In [ ]:
import json
from collections import Counter

import pandas as pd
import plotly.express as px
import yaml
from IPython.display import Markdown, display

def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

manifest = json.loads((DATASET_ROOT / "manifest.json").read_text(encoding="utf-8"))
descriptions = read_jsonl(
    DATASET_ROOT / "annotations/vlm_inferred/descriptions.jsonl"
)
verifications = read_jsonl(
    DATASET_ROOT / "annotations/vlm_inferred/verifications.jsonl"
)
facts = read_jsonl(DATASET_ROOT / "annotations/vlm_inferred/facts_all.jsonl")
trusted = read_jsonl(
    DATASET_ROOT / "annotations/vlm_inferred/facts_trusted.jsonl"
)
untrusted = read_jsonl(
    DATASET_ROOT / "annotations/vlm_inferred/facts_untrusted.jsonl"
)
catalog = json.loads(
    (DATASET_ROOT / "annotations/gt/ontology/label_catalog.json")
    .read_text(encoding="utf-8")
)
display(pd.Series(manifest["counts"]).to_frame("count"))

## 1. Изображения и уровни labels

In [ ]:
level_frame = pd.DataFrame(catalog)
observed = level_frame[level_frame.label_level != "not_observed"]
level_counts = (
    observed.groupby("label_level").size().rename("labels").reset_index()
)
display(level_counts)
fig = px.bar(
    level_counts, x="label_level", y="labels", text="labels",
    title="Наблюдавшиеся base и detailed labels",
)
fig.show()

display(
    observed[["label_id", "display_name", "label_level", "entities",
              "observed_facts", "trusted_facts"]].head(15)
)

In [ ]:
from PIL import Image

if IMAGE_ROOT is None:
    display(Markdown(
        "*Изображения не входят в metadata release на Hugging Face, поэтому "
        "visual preview в этой среде пропущен.*"
    ))
else:
    for row in descriptions[:3]:
        image_path = IMAGE_ROOT / row["image_rel_path"]
        display(Markdown(f"**{row['image_id']}**"))
        if not image_path.is_file():
            display(Markdown(f"Файл не найден: `{image_path.name}`"))
            continue
        image = Image.open(image_path).convert("RGB")
        image.thumbnail((800, 450))
        display(image)

## 2. Генерация разметки: producer → verifier

```text
image + whitelist labels + ontology
                 │
                 ▼
       GPT-5.6 Sol producer
  summary + facts[label, sentence,
        evidence, visibility]
                 │
                 ▼
      Claude Opus 5.5 verifier
 supported / uncertain / unsupported
 + faithful sentence/evidence + rationale
                 │
                 ▼
 agreed_good → trusted_fact=true
 disagreement → сохранён для аудита
```

### Почему две разные модели

- GPT-5.6 Sol использовался как producer из-за устойчивого следования JSON schema,
  работы с изображениями и annotation-constrained prompt.
- Claude Opus 5.5 использовался как независимый verifier. Другое семейство модели
  снижает вероятность одинаковых систематических ошибок producer/verifier.

Producer не мог добавлять labels за пределами whitelist существующей разметки (так как мы точно знаем что они там есть, ведь наличие labels на изображении размечено людьми, нам нужно лишь получить описания изображений).
Verifier проверял визуальную поддержку каждого факта. OCR, timestamp, watermark и
надписи извлеченные из кадров - не считались достаточным evidence.

### Исходный producer prompt

In [ ]:
producer_prompt = (DATASET_ROOT / "prompts/producer.md").read_text(encoding="utf-8")
display(Markdown(producer_prompt))

### Исходный verifier prompt

In [ ]:
verifier_prompt = (DATASET_ROOT / "prompts/verifier.md").read_text(encoding="utf-8")
display(Markdown(verifier_prompt))

## 3. Raw document, нормализованный факт и derived chunk

`annotations/vlm_inferred/descriptions.jsonl` содержит **один полный ответ на
одно изображение**.
Это исходный корпус без чанкинга.

Структура producer-ответа заранее задаёт смысловые границы: общий `description` и
массив `facts[]`. Поэтому в следующей стадии можно детерминированно получить
`one fact → one chunk`, не подбирая разделители, длину и overlap. Это преимущество
структурированного output contract, а не отдельный semantic chunker. В дальнейшем, ничего не мешает использовать стандартный чанкинг, в том числе с другими версиями датасета которые можно будет получить без структурированного output contract.

In [ ]:
example = descriptions[0]
print("image_id:", example["image_id"])
print("\nSUMMARY:\n", example["description"]["description"])
print("\nFIRST FACT:\n", json.dumps(
    example["description"]["facts"][0], ensure_ascii=False, indent=2
))

In [ ]:
fact_example = next(x for x in facts if x["image_id"] == example["image_id"])
display(pd.Series({
    "fact_id": fact_example["fact_id"],
    "label_id": fact_example["label_id"],
    "sentence": fact_example["sentence"],
    "evidence": fact_example["evidence"],
    "verdict": fact_example["verifier"]["verdict"],
    "rationale": fact_example["verifier"]["rationale"],
    "trusted_fact": fact_example["trusted_fact"],
}).to_frame("value"))

## 4. Доверенные и недоверенные факты

In [ ]:
state_counts = pd.DataFrame(
    Counter(x["agreement_state"] for x in facts).items(),
    columns=["agreement_state", "facts"],
).sort_values("facts", ascending=False)
display(state_counts)
fig = px.bar(
    state_counts, x="agreement_state", y="facts", color="agreement_state",
    title="Результат producer-verifier agreement",
)
fig.show()

`trusted_fact=true` означает, что verifier подтвердил категорию и faithful wording
sentence/evidence. Недоверенные факты не удаляются. Отсутствующая категория остаётся
`unjudged`, поэтому её нельзя автоматически использовать как negative example.

## 5. EDA исходных 500 описаний

In [ ]:
from gigaschool_rag.versions.v0.eda import analyze_dataset

show_package_source(analyze_dataset)
eda_result = analyze_dataset(DATASET_ROOT)
documents = eda_result["documents"]
lengths = eda_result["lengths"]
display(pd.DataFrame(lengths).T)

Полный raw JSON проходит требование 200k токенов. Проекция
`summary + facts + evidence` также превышает 200k токенов, но технический JSON
остаётся отдельно, чтобы не смешивать metadata и retrieval text.

In [ ]:
plot_frame = documents.melt(
    id_vars=["image_id"],
    value_vars=["raw_tokens", "summary_tokens", "semantic_tokens"],
    var_name="projection", value_name="tokens",
)
fig = px.histogram(
    plot_frame, x="tokens", color="projection", barmode="overlay", nbins=60,
    title="Распределение длины документов",
)
fig.add_vline(x=50, line_dash="dash")
fig.add_vline(x=2000, line_dash="dash")
fig.show()

In [ ]:
noise = eda_result["noise"]
duplicates = eda_result["duplicates"]
display(pd.Series(noise).to_frame("noise"))
display(pd.Series(duplicates).to_frame("duplicates"))

In [ ]:
clusters = eda_result["clusters"]
display(pd.DataFrame(clusters["silhouette_candidates"]))
display(pd.DataFrame(clusters["clusters"]))

### Выводы EDA

- Типичный semantic document содержит несколько сотен токенов; raw JSON длиннее
  из-за whitelist, provenance и usage.
- Основной шум — IDs, числа, JSON-пунктуация и служебные поля, а не URL/emoji.
- Exact и near duplicates общих summaries практически отсутствуют; повторяемость
  появляется на уровне отдельных категорий/facts.
- Тематические кластеры отражают транспорт, погоду, здания, положение камеры и
  качество изображения.
- Для следующих экспериментов разумно сравнить image-parent documents, fact
  children и free-form fixed-size chunks.

## 6. Канонический gold-набор v0

Фиксированный benchmark из 50 вопросов хранится как versioned package resource внутри `gigaschool_rag.versions.v0`. Поэтому notebook получает одинаковые validation/test splits локально и в Colab, но не зависит от физического расположения `evaluation/` в checkout. Gold не строится из результатов текущего retriever.

In [ ]:
from gigaschool_rag.versions.v0.evaluation import load_gold

show_package_source(load_gold)
gold = load_gold()
gold_frame = pd.DataFrame(gold)
display(
    gold_frame.groupby(["split", "question_type"]).size()
    .rename("questions").reset_index()
)
print("Всего:", len(gold))
print("Validation:", sum(gold_frame.split == "validation"))
print("Test:", sum(gold_frame.split == "test"))
print("С намеренным шумом:", int(gold_frame.contains_intentional_noise.sum()))

In [ ]:
from gigaschool_rag.versions.v0.evaluation import load_rejected_candidates

show_package_source(load_rejected_candidates)
rejected = load_rejected_candidates()
display(pd.DataFrame(rejected))

### Как формировались категории вопросов

Gold содержит четыре типа вопросов, проверяющих разные режимы retrieval:

- **15 factual** — запросы по одной конкретной категории. Кандидаты выбирались среди наиболее частых labels с не менее чем пятью trusted facts. Слишком тривиальные и доминирующие `day` и `night` исключались. Формулировки чередуют formal и conversational templates, чтобы benchmark не зависел от одного шаблона.
- **15 composite** — конъюнктивные запросы по двум labels. Использовались только пары, которые действительно совместно встречаются как trusted facts минимум на трёх изображениях. Labels пары выбирались из разных ontology entities, поэтому вопрос требует пересечения двух признаков, а не повторения близких синонимов. Из подходящих пар брались 15 наиболее частых.
- **10 abstract** — широкие пользовательские темы, которые нельзя выразить одной категорией: road activity, pedestrian infrastructure, public transport, parking infrastructure, adverse weather, clear/sunny weather, rail infrastructure, commercial places, residential environment и image-quality problems. Для каждой темы вручную задан прозрачный набор ontology labels; релевантность является объединением trusted изображений этих labels.
- **10 negative** — запросы по ontology labels, для которых в pilot нет ни одного trusted fact. Их relevance set намеренно пуст: они проверяют ложные срабатывания и способность retrieval не выдавать неподтверждённые изображения. Поэтому такие вопросы помечены как `answerable=false`.

В семь вопросов (14%) детерминированно добавлены опечатки или разговорные формы для небольшой проверки устойчивости к шуму: два factual, два composite, один abstract и два negative. При построении проверялось отсутствие неизвестных image/fact IDs, непустая релевантность положительных вопросов, пустая релевантность negative и отсутствие служебных identifier leaks в тексте запроса.

Validation/test сформированы **стратифицированно по типу вопроса** с фиксированным seed: в test заморожены 5 factual, 4 composite, 3 abstract и 3 negative вопроса; оставшиеся 35 относятся к validation. Ground truth ссылается на image IDs и нормализованные trusted fact IDs.

## 7. Почему No-RAG baseline не применяется

Постановка задачи требует вернуть идентификаторы конкретных изображений из закрытого корпуса по визуальным объектам и условиям в кадре. Без retrieval-контекста модель не получает:

- сами изображения;
- их текстовые описания и trusted facts;
- множество доступных `image_id`;
- соответствие между `image_id` и содержимым кадра.

Следовательно, даже модель с широкими общими знаниями принципиально не может определить, какие внутренние изображения релевантны вопросу. Единственное корректное поведение без контекста — отказаться от ответа и не выдумывать идентификаторы. Нулевой recall в таком режиме является прямым следствием постановки задачи, а не экспериментально интересной характеристикой модели.

Поэтому отдельный LLM No-RAG запуск и программный baseline здесь не выполняются: они добавили бы стоимость и недетерминизм, но не дали бы содержательного сравнения. Практически полезная нижняя граница начинается с retrieval по корпусу — например, с TF-IDF baseline в следующей секции.

## 8. Evidence предсказуемого fact-level chunking

In [ ]:
from gigaschool_rag.versions.v0.evaluation import (
    evaluate_gold_retrieval,
    evaluate_queries,
)

show_package_source(evaluate_gold_retrieval, evaluate_queries)
retrieval = evaluate_gold_retrieval(DATASET_ROOT, gold=gold)
display(pd.Series({
    "retriever": retrieval["retriever"],
    "queries": retrieval["queries"],
    "chunks": retrieval["chunks"],
    "images": retrieval["images"],
    "mAP": retrieval["mean_average_precision"],
}).to_frame("value"))
display(pd.DataFrame(retrieval["metrics_by_k"]))

TF-IDF на детерминированных `one fact → one chunk` даёт измеримое evidence, что
структурированный output хорошо индексируется: границы не зависят от эвристики по
свободному тексту, а каждый chunk связан с parent image и label.

Но это **не доказательство превосходства** над free-form descriptions:

- label wording может встречаться в sentence/evidence;
- отсутствует matched A/B baseline со свободным описанием;
- TF-IDF особенно чувствителен к lexical overlap.

Корректный вывод: structured facts являются сильным и воспроизводимым baseline.
Проверка гипотезы «лучше free-form» требует следующего эксперимента на одинаковых
queries, splits, embedding model и retrieval budget.

## 9. Итог

В `v0` разделены три слоя:

1. `annotations/gt/` — исходная human GT-разметка и ontology;
2. `annotations/vlm_inferred/` — полные producer/verifier outputs и trust decisions;
3. `offline_chunking/` — регенерируемые fact chunks.

Такое разделение сохраняет исходные данные для EDA, позволяет менять trust policy
и одновременно использует структурированный output contract для предсказуемого
чанкинга в последующих контрольных точках.